# Bone pair alignment (PCA + ICP)

One function: two bone meshes in, best RMS out.

Pipeline (no picking, no folder loop):
1. Sample points from each mesh
2. Center and unit-scale (centroid size)
3. PCA so PC1 / PC2 / PC3 line up with XYZ
4. Try all 4 proper **and** 4 improper (mirrored) sign-flips — both chiralities
5. For each of those 8, roll around PC1 at 30° steps (0, 30, …, 330) — 96 candidates
6. Short ICP on each; return the **lowest** RMS

PC1 rolls cover leftover twist around the long axis (and make a PC2/PC3 swap unnecessary). 360° is the same as 0°, so it is not included. Batch automation and assessment come later.

## Install Open3D

`open3d` is a separate Python package (not part of CloudCompare). Install it into **the same Python this notebook kernel uses**, then **restart the kernel** and re-run the import cell:

```bash
python -m pip install numpy open3d
```

If the import still fails, the kernel is a different interpreter. Run `import sys; print(sys.executable)` here, then:

```bash
/that/python -m pip install open3d
```

`ModuleNotFoundError: No module named 'open3d'` is expected until that install and restart.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import numpy as np
import open3d as o3d

In [ ]:
# Right-handed sign flips of XYZ (det = +1).
_PROPER_FLIPS = (
    np.diag([1.0, 1.0, 1.0]),
    np.diag([1.0, -1.0, -1.0]),
    np.diag([-1.0, 1.0, -1.0]),
    np.diag([-1.0, -1.0, 1.0]),
)

# Improper (reflections, det = -1) — left vs right after both are in PCA frames.
_IMPROPER_FLIPS = (
    np.diag([1.0, 1.0, -1.0]),
    np.diag([1.0, -1.0, 1.0]),
    np.diag([-1.0, 1.0, 1.0]),
    np.diag([-1.0, -1.0, -1.0]),
)


def _as_points(bone: Any, n_samples: int, rng: np.random.Generator) -> np.ndarray:
    """Load a path, Open3D/Trimesh mesh, or (N, 3) array into sampled points."""
    if isinstance(bone, (str, Path)):
        path = str(bone)
        mesh = o3d.io.read_triangle_mesh(path)
        if mesh.has_triangles() and len(mesh.triangles) > 0:
            pcd = mesh.sample_points_uniformly(number_of_points=n_samples)
            pts = np.asarray(pcd.points, dtype=float)
            if pts.size == 0:
                raise ValueError(f"No points sampled from mesh: {path}")
            return pts
        pcd = o3d.io.read_point_cloud(path)
        pts = np.asarray(pcd.points, dtype=float)
        if pts.size == 0:
            raise ValueError(f"Could not load points from: {path}")
        return _subsample(pts, n_samples, rng)

    if isinstance(bone, np.ndarray):
        return _subsample(np.asarray(bone, dtype=float), n_samples, rng)

    if isinstance(bone, o3d.geometry.TriangleMesh):
        pcd = bone.sample_points_uniformly(number_of_points=n_samples)
        return np.asarray(pcd.points, dtype=float)

    if isinstance(bone, o3d.geometry.PointCloud):
        return _subsample(np.asarray(bone.points, dtype=float), n_samples, rng)

    if hasattr(bone, "sample_points_uniformly"):
        pcd = bone.sample_points_uniformly(number_of_points=n_samples)
        return np.asarray(pcd.points, dtype=float)

    if hasattr(bone, "vertices"):
        verts = np.asarray(bone.vertices, dtype=float)
        return _subsample(verts, n_samples, rng)

    raise TypeError(f"Unsupported bone type: {type(bone)!r}")


def _subsample(pts: np.ndarray, n_samples: int, rng: np.random.Generator) -> np.ndarray:
    pts = np.asarray(pts, dtype=float).reshape(-1, 3)
    if len(pts) > n_samples:
        idx = rng.choice(len(pts), n_samples, replace=False)
        pts = pts[idx]
    return pts


def _center_and_unit_scale(pts: np.ndarray) -> np.ndarray:
    centered = pts - pts.mean(axis=0)
    size = float(np.sqrt(np.mean(np.sum(centered * centered, axis=1))))
    if size < 1e-12:
        raise ValueError("Degenerate point set (centroid size is ~0)")
    return centered / size


def _pca_frame(pts: np.ndarray) -> np.ndarray:
    """Rotate a centered cloud so PC1, PC2, PC3 align with XYZ (right-handed)."""
    cov = np.cov(pts, rowvar=False)
    evals, evecs = np.linalg.eigh(cov)
    order = np.argsort(evals)[::-1]
    R = evecs[:, order]
    if np.linalg.det(R) < 0:
        R[:, 2] *= -1
    return pts @ R


def _to_pcd(pts: np.ndarray) -> o3d.geometry.PointCloud:
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(np.ascontiguousarray(pts, dtype=float))
    return pcd


def _full_rmse(src: np.ndarray, dst_pcd: o3d.geometry.PointCloud, T: np.ndarray) -> float:
    homo = np.c_[src, np.ones(len(src))]
    aligned = (T @ homo.T).T[:, :3]
    dists = _to_pcd(aligned).compute_point_cloud_distance(dst_pcd)
    dists = np.asarray(dists)
    return float(np.sqrt(np.mean(dists * dists)))


def _rotation_about_pc1(degrees: float) -> np.ndarray:
    """Rotation about PC1 (X) after the PCA frame."""
    t = np.deg2rad(degrees)
    c, s = np.cos(t), np.sin(t)
    return np.array(
        [
            [1.0, 0.0, 0.0],
            [0.0, c, -s],
            [0.0, s, c],
        ]
    )


def _pc1_angles(step_deg: float) -> np.ndarray:
    if step_deg <= 0 or step_deg > 360:
        raise ValueError(f"pc1_step_deg must be in (0, 360], got {step_deg}")
    return np.arange(0.0, 360.0, step_deg)


def _candidate_bases(pc1_step_deg: float = 30.0) -> list[np.ndarray]:
    angles = _pc1_angles(pc1_step_deg)
    bases: list[np.ndarray] = []
    for S in (*_PROPER_FLIPS, *_IMPROPER_FLIPS):
        for deg in angles:
            bases.append(S @ _rotation_about_pc1(float(deg)))
    return bases

In [ ]:
def align_bones(
    bone_a,
    bone_b,
    *,
    n_samples: int = 20000,
    pc1_step_deg: float = 30.0,
    icp_max_corr: float = 0.25,
    icp_iterations: int = 80,
    seed: int = 42,
) -> float:
    """Align two bones (PCA + sign-flips + PC1 rolls + ICP) and return the best RMS.

    Searches both chiralities (proper and mirrored sign-flips) and rolls around
    PC1 at ``pc1_step_deg`` intervals (default 30° → 0, 30, …, 330).

    Parameters
    ----------
    bone_a : path, mesh, or (N, 3) array
        Moving bone.
    bone_b : path, mesh, or (N, 3) array
        Fixed bone (reference).
    n_samples : int
        Points sampled from each mesh for PCA/ICP.
    pc1_step_deg : float
        Roll step around PC1 in degrees. 30 → 12 rolls × 8 bases = 96 candidates.
        360° is omitted (same as 0°).
    icp_max_corr : float
        ICP correspondence distance in unit-scale space (centroid size = 1).
    icp_iterations : int
        Max ICP iterations per candidate.
    seed : int
        RNG seed for subsampling.

    Returns
    -------
    float
        Lowest RMS (root-mean-square nearest-neighbor distance) after ICP.
        Because clouds are unit-scaled, this is relative to centroid size.
    """
    rng = np.random.default_rng(seed)
    pts_a = _pca_frame(_center_and_unit_scale(_as_points(bone_a, n_samples, rng)))
    pts_b = _pca_frame(_center_and_unit_scale(_as_points(bone_b, n_samples, rng)))

    dst = _to_pcd(pts_b)
    criteria = o3d.pipelines.registration.ICPConvergenceCriteria(
        max_iteration=icp_iterations
    )
    estimation = o3d.pipelines.registration.TransformationEstimationPointToPoint()

    best_rms = np.inf
    for S in _candidate_bases(pc1_step_deg):
        src_pts = pts_a @ S
        src = _to_pcd(src_pts)
        reg = o3d.pipelines.registration.registration_icp(
            src,
            dst,
            icp_max_corr,
            np.eye(4),
            estimation,
            criteria,
        )
        rms = _full_rmse(src_pts, dst, np.asarray(reg.transformation))
        if rms < best_rms:
            best_rms = rms

    return float(best_rms)

## Try a pair

Point `left` and `right` at two mesh files (`.stl`, `.ply`, `.obj`, …). RMS is in unit-scale (centroid size of each bone is 1), so values around a few hundredths are a tight fit; much larger usually means a bad candidate won or the meshes are not homologous.

Default search: both flipped and non-flipped bases, plus a 30° PC1 roll on each (96 ICP runs). Pass `pc1_step_deg=15` for a finer sweep if a pair is still off.

In [ ]:
left = "path/to/left_bone.stl"
right = "path/to/right_bone.stl"

# rms = align_bones(left, right)  # both chiralities + 30° PC1 rolls
# print(rms)